<a href="https://colab.research.google.com/github/aldikayyis/Pemantauan-Risiko-Bencana-Gempa-Bumi-Terkini-Studi-Kasus-KSP---Deputi-II-/blob/main/Visualisasi_Peta_Risiko_Gempa_BMKG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster
import os

# ==========================================
# 1. SETUP & LOAD DATA
# ==========================================
# Ganti dengan nama file CSV hasil scraping lo sebelumnya
FILE_CSV = 'data_gempa_terkini_bmkg.csv'

def load_data(file_path):
    print(f"📖 Membaca data dari: {file_path}")
    try:
        df = pd.read_csv(file_path)

        # Validasi kolom wajib untuk peta
        kolom_wajib = ['Lintang', 'Bujur', 'Magnitude', 'Kedalaman', 'Wilayah', 'Tanggal', 'Jam']
        for col in kolom_wajib:
            if col not in df.columns:
                print(f"❌ Error: Kolom '{col}' tidak ditemukan di CSV!")
                return None
        return df
    except FileNotFoundError:
        print(f"❌ Error: File {file_path} tidak ditemukan. Pastikan script scraper BMKG sudah dijalankan.")
        return None

# ==========================================
# 2. LOGIKA KLASIFIKASI RISIKO (BUSINESS RULES)
# ==========================================
def tentukan_warna_kedalaman(kedalaman_str):
    """
    Mengklasifikasikan warna marker berdasarkan kedalaman gempa.
    Secara geologis:
    - Dangkal (< 60 km): Merah (Lebih merusak permukaan)
    - Menengah (60 - 300 km): Oranye
    - Dalam (> 300 km): Hijau
    """
    try:
        # Menghapus kata ' km' dan ubah ke integer (misal: "10 km" -> 10)
        kedalaman = int(str(kedalaman_str).lower().replace(' km', '').strip())

        if kedalaman <= 60:
            return 'red'
        elif kedalaman <= 300:
            return 'orange'
        else:
            return 'green'
    except ValueError:
        return 'gray' # Warna default jika data kedalaman error

# ==========================================
# 3. PEMBUATAN PETA (MAPPING)
# ==========================================
def buat_peta_interaktif(df):
    print("🗺️ Membuat peta interaktif...")

    # Inisiasi Peta Dasar (Pusat titik tengah di tengah-tengah Indonesia)
    # Titik tengah kasar Indonesia: Lintang -2.5, Bujur 118.0
    peta_indonesia = folium.Map(
        location=[-2.5, 118.0],
        zoom_start=5,
        tiles='CartoDB positron' # Tema peta yang bersih dan profesional
    )

    # Menambahkan layer judul/keterangan di atas peta
    title_html = '''
             <h3 align="center" style="font-size:20px; font-family:Arial; margin-top:20px;"><b>
             Dashboard Pemantauan Risiko Gempabumi Terkini (BMKG)
             </b></h3>
             '''
    peta_indonesia.get_root().html.add_child(folium.Element(title_html))

    # Iterasi per baris data CSV
    for index, row in df.iterrows():
        # Lewati jika ada data koordinat yang kosong (NaN)
        if pd.isna(row['Lintang']) or pd.isna(row['Bujur']):
            continue

        lat = row['Lintang']
        lon = row['Bujur']
        mag = float(row['Magnitude'])
        wilayah = row['Wilayah']
        waktu = f"{row['Tanggal']} | {row['Jam']}"
        kedalaman = row['Kedalaman']
        potensi = row.get('Potensi', 'Tidak ada data potensi')

        # Tentukan estetika marker
        warna = tentukan_warna_kedalaman(kedalaman)
        # Radius marker membesar secara eksponensial seiring naiknya magnitude
        radius = (mag - 4) * 5 if mag > 4 else 5

        # HTML untuk Popup yang profesional (Mirip dashboard BI)
        popup_html = f"""
        <div style="width:250px; font-family:Arial; font-size:12px;">
            <h4 style="margin-bottom:5px; color:#2c3e50;"><b>{wilayah}</b></h4>
            <hr style="margin:5px 0;">
            <b>Waktu:</b> {waktu}<br>
            <b>Magnitude:</b> <span style="color:red; font-size:14px;"><b>{mag} SR</b></span><br>
            <b>Kedalaman:</b> <span style="color:{warna};"><b>{kedalaman}</b></span><br>
            <b>Potensi:</b> {potensi}<br>
            <b>Koordinat:</b> {lat}, {lon}
        </div>
        """

        # Tambahkan marker berbentuk lingkaran ke peta
        folium.CircleMarker(
            location=[lat, lon],
            radius=radius,
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=f"Mag {mag} - {wilayah}", # Teks saat kursor di-hover
            color=warna,
            fill=True,
            fill_color=warna,
            fill_opacity=0.6,
            weight=1
        ).add_to(peta_indonesia)

    # Simpan peta sebagai file HTML
    nama_file_peta = 'dashboard_peta_gempa.html'
    peta_indonesia.save(nama_file_peta)

    print(f"✅ Selesai! Peta berhasil disimpan sebagai: {os.path.abspath(nama_file_peta)}")
    print("\n🚀 CARA MELIHAT PETA:")
    print(f"Buka file '{nama_file_peta}' yang baru saja terbuat menggunakan web browser (Chrome/Edge/Firefox).")
    print("Atau di Colab, lu bisa download file html tersebut dari panel sebelah kiri.")

# ==========================================
# EKSEKUSI
# ==========================================
if __name__ == "__main__":
    df_gempa = load_data(FILE_CSV)

    if df_gempa is not None:
        buat_peta_interaktif(df_gempa)

📖 Membaca data dari: data_gempa_terkini_bmkg.csv
🗺️ Membuat peta interaktif...
✅ Selesai! Peta berhasil disimpan sebagai: /content/dashboard_peta_gempa.html

🚀 CARA MELIHAT PETA:
Buka file 'dashboard_peta_gempa.html' yang baru saja terbuat menggunakan web browser (Chrome/Edge/Firefox).
Atau di Colab, lu bisa download file html tersebut dari panel sebelah kiri.
